In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler

In [2]:
data= pd.read_csv("../data/model_data.csv",index_col=0)
print(data.shape)

(1798, 11)


In [3]:
y= data['churned']
X=data.drop(columns='churned')
print(X.shape,y.shape)

(1798, 10) (1798,)


In [6]:

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(1438, 10) (360, 10)
0.2997218358831711 0.3


In [9]:
dummy = DummyClassifier(strategy='most_frequent') 
dummy.fit(X_train,y_train)
dummy_pred=dummy.predict(X_test)
print(dummy_pred[:10])
print((dummy_pred == y_test).mean())

[0 0 0 0 0 0 0 0 0 0]
0.7


In [10]:
from sklearn.metrics import classification_report
print(classification_report(y_test,dummy_pred))

              precision    recall  f1-score   support

           0       0.70      1.00      0.82       252
           1       0.00      0.00      0.00       108

    accuracy                           0.70       360
   macro avg       0.35      0.50      0.41       360
weighted avg       0.49      0.70      0.58       360



c:\Users\USER\Desktop\Mohamad Fares Projects\churn-prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\Desktop\Mohamad Fares Projects\churn-prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\Desktop\Mohamad Fares Projects\churn-prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to co

In [13]:

scaler= StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
print(X_train_scaled.mean(axis=0).round(2))
print(X_test_scaled.mean(axis=0).round(2))

[-0.  0.  0. -0. -0.  0.  0.  0.  0. -0.]
[-0.01  0.01  0.02 -0.02 -0.   -0.06  0.03 -0.   -0.    0.01]


In [14]:
from sklearn.linear_model import LogisticRegression
logreg= LogisticRegression(random_state=42)
logreg.fit(X_train_scaled,y_train)
logreg_pred=logreg.predict(X_test_scaled)
print(classification_report(y_test,logreg_pred))


              precision    recall  f1-score   support

           0       0.75      0.90      0.82       252
           1       0.55      0.30      0.39       108

    accuracy                           0.72       360
   macro avg       0.65      0.60      0.60       360
weighted avg       0.69      0.72      0.69       360



In [15]:
logreg_proba = logreg.predict_proba(X_test_scaled)[:, 1]
print(logreg_proba[:5].round(2))

[0.47 0.3  0.4  0.59 0.01]


In [16]:
new_pred =(logreg_proba>0.3).astype(int)
print(classification_report(y_test,new_pred))

              precision    recall  f1-score   support

           0       0.85      0.68      0.76       252
           1       0.49      0.71      0.58       108

    accuracy                           0.69       360
   macro avg       0.67      0.70      0.67       360
weighted avg       0.74      0.69      0.70       360



In [17]:
from sklearn.metrics import roc_auc_score
print(roc_auc_score(y_test,logreg_proba))

0.7578630217519107


In [18]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(n_estimators=200,random_state=42)
rf.fit(X_train,y_train)
rf_proba=rf.predict_proba(X_test)[:,1]
print(roc_auc_score(y_test,rf_proba))

0.7206973838918282


In [19]:
print(roc_auc_score(y_train, rf.predict_proba(X_train)[:, 1]))

1.0


In [20]:
rf2=RandomForestClassifier(n_estimators=200,max_depth=5,min_samples_leaf=10,random_state=42)
rf2.fit(X_train,y_train)
print (roc_auc_score(y_train,rf2.predict_proba(X_train)[:,1]))
print (roc_auc_score(y_test,rf2.predict_proba(X_test)[:,1]))


0.84997592260211
0.7598838918283364


In [21]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(estimator=rf2, X=X_train, y=y_train, cv=5, scoring='roc_auc')
print(scores.round(3),scores.mean().round(3))

[0.788 0.773 0.747 0.782 0.774] 0.773


In [28]:
from sklearn.pipeline import make_pipeline
logreg_pipe=make_pipeline(StandardScaler(),LogisticRegression(random_state=42))
scores_lr=cross_val_score(estimator=logreg_pipe,X=X_train,y=y_train,cv=5,scoring="roc_auc")
print(scores_lr.round(3),scores_lr.mean().round(4))

[0.788 0.745 0.747 0.761 0.774] 0.7632


In [23]:
from sklearn.model_selection import GridSearchCV
param_grid={"max_depth":[3,5,8], "min_samples_leaf":[5,10,20]} 
grid=GridSearchCV(estimator= RandomForestClassifier(n_estimators=200,random_state=42),param_grid=param_grid,cv=5,scoring="roc_auc")
grid.fit(X_train,y_train)
print(grid.best_params_,grid.best_score_.round(3))

{'max_depth': 8, 'min_samples_leaf': 20} 0.773


In [25]:
rf_best=grid.best_estimator_
imp=pd.Series(rf_best.feature_importances_,index=X_train.columns).sort_values(ascending=False)
print(imp.round(3))

recency          0.200
recent_spend     0.161
usual_gap        0.149
monetary         0.114
frequency        0.078
spend_share      0.074
variety          0.072
avg_order        0.062
recent_orders    0.046
recent_share     0.045
dtype: float64


In [27]:
X_small = X_train.drop(columns=["recent_orders", "recent_share", "avg_order"])
scores_small = cross_val_score(estimator=rf_best, X=X_small, y=y_train, cv=5, scoring="roc_auc")
print(scores_small.mean().round(4))


0.7725


In [29]:
final_proba = rf_best.predict_proba(X_test)[:, 1]
print(roc_auc_score(y_test, final_proba))

0.7540417401528513


In [30]:
from sklearn.model_selection import cross_val_predict
oof_proba= cross_val_predict(rf_best,X_train,y_train,cv=5,method="predict_proba")[:,1]
print(oof_proba[:5].round(2))

[0.34 0.42 0.48 0.35 0.16]


In [32]:
from sklearn.metrics import recall_score,precision_score
for t in [0.2,0.25,0.3,0.35,0.4,0.45,0.5]:
    pred=(oof_proba>t).astype(int)
    print(t, round(recall_score(y_train, pred), 2), round(precision_score(y_train, pred), 2))

0.2 0.89 0.42
0.25 0.83 0.44
0.3 0.76 0.47
0.35 0.66 0.49
0.4 0.55 0.52
0.45 0.44 0.59
0.5 0.35 0.66


In [33]:
final_pred = (final_proba > 0.3).astype(int)
print(classification_report(y_test, final_pred))

              precision    recall  f1-score   support

           0       0.85      0.63      0.73       252
           1       0.47      0.75      0.57       108

    accuracy                           0.67       360
   macro avg       0.66      0.69      0.65       360
weighted avg       0.74      0.67      0.68       360



In [34]:
bundle = {"model": rf_best, "threshold": 0.3, "features": list(X_train.columns)}

In [35]:
import joblib
joblib.dump(bundle, "../models/churn_model.joblib")

['../models/churn_model.joblib']

In [36]:
loaded = joblib.load("../models/churn_model.joblib")
print(loaded["threshold"], loaded["features"])

0.3 ['recency', 'frequency', 'monetary', 'avg_order', 'variety', 'usual_gap', 'recent_orders', 'recent_share', 'recent_spend', 'spend_share']
